# Lab 11B — Off-Policy Evaluation on the Tennessee Eastman Simulator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sreent/causal-ai-for-smart-manufacturing/blob/main/labs/ch11/lab11b.ipynb)

**Companion to Lab 11A.** Lab 11A built a tabular MDP where we knew both the behaviour policy and the true value of the evaluation policy, then verified that IPS, SNIPS, and DR each recover that value within sampling error. **Lab 11B asks the same question on a continuous-state chemical-process simulator — the Tennessee Eastman (TE) plant.**

The setup is one a process-control engineer would recognise: someone logged a trajectory under the *current* operating recipe (a stochastic policy on a single intervention switch), a candidate *new* policy is proposed, and we have to estimate the candidate's expected reward *without* deploying it. The deliverable is the OPE estimate, the variance diagnostic (effective sample size), and a deploy-or-pilot decision.

**Dataset.** Two pre-simulated TE trajectories, generated by the vendored `tep2py` Fortran simulator (Downs & Vogel 1993) and committed in the repo:

- **Logged** (behaviour policy): 500 samples with the IDV(1) ("A/C Feed Ratio") disturbance toggled Bernoulli(0.5) each 3-minute step.
- **Candidate** (evaluation policy): 500 samples with IDV(1) = 0 always — i.e., never trigger the disturbance.

Because the simulator is the *true* generating process, the candidate trajectory's empirical return *is* the ground truth value of the evaluation policy. OPE on the logged data should recover it (within finite-sample noise). If it doesn't, the OPE estimator is biased — and that bias is the lesson.

## What this lab is *not* doing

- **Building a sequential policy.** The evaluation policy is *stationary and deterministic* (always action = 0). Sequential / state-conditioned policies require the trajectory-product IPS form and substantially more data; we use the simpler per-step IPS suited to a stationary $\pi_e$.
- **Running the live simulator.** The two trajectories are pre-generated. Re-simulation requires building the Fortran extension (see `labs/data/te_simulator/README.md`); the canonical CSVs cover this lab without it.
- **Tuning the reward.** The reward is a single, fixed quadratic deviation from a target reactor pressure. A real engineering analysis would use a multi-objective cost (pressure + temperature + quality + energy).

In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn

In [ ]:
import os, sys, urllib.request, pathlib

# Pull the TE prep module from the repo on first run; it loads the committed CSVs.
PREP_PATH = pathlib.Path("/content/te_prep.py")
if not PREP_PATH.exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/sreent/causal-ai-for-smart-manufacturing/main/labs/data/te_prep.py",
        PREP_PATH,
    )
sys.path.insert(0, str(PREP_PATH.parent))

# The loader resolves CSVs relative to its own directory. Mirror the repo layout
# under /content so the loader finds them.
for name in ("te_logged.csv", "te_candidate.csv"):
    p = pathlib.Path("/content") / name
    if not p.exists():
        urllib.request.urlretrieve(
            f"https://raw.githubusercontent.com/sreent/causal-ai-for-smart-manufacturing/main/labs/data/{name}",
            p,
        )

from te_prep import load_te

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor

rng = np.random.default_rng(0)

## Part 1 — Load both trajectories

In [ ]:
log  = load_te("logged")
cand = load_te("candidate")

print(f"Logged trajectory:    {log.shape}   action rate (IDV1=1): {log['action_idv1'].mean():.2f}")
print(f"Candidate trajectory: {cand.shape}   action rate (IDV1=1): {cand['action_idv1'].mean():.2f}")
print()
print("Logged columns (first 6):", list(log.columns)[:6], "...")

## Part 2 — Frame the OPE problem

**State $s_t$.** The TE process measurements at step $t$ (the 41 XMEAS variables).

**Action $a_t$.** The per-step value of IDV(1) — 0 or 1.

**Reward $r_t$.** Negative squared deviation of reactor pressure (XMEAS(7)) from a target of 2700 kPa, which is the published normal operating point for TE. This is a *cost-minimisation* framing: $r_t = -(XMEAS(7)_t - 2700)^2 / 10^6$ (scaled for readability).

**Behaviour policy $\pi_b$.** $\pi_b(a_t = 1 \mid s_t) = 0.5$ for all $t$ (uniform random over the action toggle). This is by construction of the logged trajectory.

**Evaluation policy $\pi_e$.** $\pi_e(a_t = 0 \mid s_t) = 1$ for all $t$ (always off). This is by construction of the candidate trajectory.

**Estimand.** $V(\pi_e) = E_{\pi_e}[r_t]$ — the expected per-step reward under the evaluation policy.

In [ ]:
def reward(df, pressure_target=2700.0):
    return -((df["XMEAS(7)"].values - pressure_target) ** 2) / 1e6

r_log  = reward(log)
r_cand = reward(cand)

print(f"Logged  mean reward (mixed actions):     {r_log.mean():+.4f}")
print(f"Candidate mean reward (always off):       {r_cand.mean():+.4f}")
print()
print(f"Ground-truth V(pi_e) (from candidate):   {r_cand.mean():+.4f}")
print("This is the number OPE must recover from the logged data alone.")

## Part 3 — Per-step IPS

For a stationary, deterministic $\pi_e$ and a stationary stochastic $\pi_b$, per-step IPS is:

$$\hat{V}_{IPS}(\pi_e) = \frac{1}{N} \sum_{t=1}^{N} \frac{\pi_e(a_t \mid s_t)}{\pi_b(a_t \mid s_t)} \cdot r_t.$$

For our setup, $\pi_e(0 \mid s_t) = 1$ and $\pi_b(0 \mid s_t) = 0.5$, so the importance weight is $2$ on steps where $a_t = 0$ and $0$ elsewhere.

In [ ]:
def importance_weights(actions, pe_action=0, pb_prob=0.5):
    """Per-step weight for a deterministic pi_e and a Bernoulli(pb_prob) pi_b."""
    return np.where(actions == pe_action, 1.0 / pb_prob, 0.0)

w = importance_weights(log["action_idv1"].values, pe_action=0, pb_prob=0.5)
V_ips = float(np.mean(w * r_log))

# Effective sample size (Kish): a variance diagnostic; small ESS -> high-variance IPS.
ess = (w.sum() ** 2) / (w ** 2).sum()
ess_frac = ess / len(w)

print(f"IPS estimate of V(pi_e):                 {V_ips:+.4f}")
print(f"Effective sample size (Kish):            {ess:.0f} of {len(w)}  ({100*ess_frac:.1f}%)")
print(f"Ground truth V(pi_e):                    {r_cand.mean():+.4f}")
print(f"IPS bias vs ground truth:                {V_ips - r_cand.mean():+.4f}")

## Part 4 — Self-normalised IPS (SNIPS)

SNIPS divides by the sum of weights instead of $N$, trading a small bias for substantially lower variance:

$$\hat{V}_{SNIPS}(\pi_e) = \frac{\sum_t w_t \cdot r_t}{\sum_t w_t}.$$

In [ ]:
V_snips = float(np.sum(w * r_log) / np.sum(w))

print(f"SNIPS estimate of V(pi_e):               {V_snips:+.4f}")
print(f"Ground truth V(pi_e):                    {r_cand.mean():+.4f}")
print(f"SNIPS bias vs ground truth:              {V_snips - r_cand.mean():+.4f}")

## Part 5 — Doubly robust (DR)

DR augments IPS with a fitted reward model $\hat{r}(s, a)$. The variance reduction comes from using the model to predict the reward at the *evaluation* action even on steps where the logged action differed.

$$\hat{V}_{DR}(\pi_e) = \frac{1}{N} \sum_t \left[\hat{r}(s_t, \pi_e(s_t)) + \frac{\pi_e(a_t \mid s_t)}{\pi_b(a_t \mid s_t)} \cdot (r_t - \hat{r}(s_t, a_t))\right].$$

If $\hat{r}$ is exactly right, the IPS correction term has mean zero and DR equals the model-based estimate. If $\pi_b$ is exactly right, the IPS correction recovers truth even under model misspecification. DR is consistent if *either* component is correct — the *doubly* in doubly robust.

In [ ]:
state_cols = [c for c in log.columns if c.startswith("XMEAS")]
S_log = log[state_cols].values
a_log = log["action_idv1"].values

# Fit r_hat(s, a) as a single regressor with a as a feature.
features = np.hstack([S_log, a_log.reshape(-1, 1)])
r_model = GradientBoostingRegressor(random_state=0)
r_model.fit(features, r_log)

# Direct model term: r_hat(s, pi_e(s)) — pi_e always picks a=0.
features_pe = np.hstack([S_log, np.zeros((len(S_log), 1))])
direct = r_model.predict(features_pe)

# IPS correction term.
r_hat_at_logged = r_model.predict(features)
correction = w * (r_log - r_hat_at_logged)

V_dr = float(np.mean(direct + correction))

print(f"DR estimate of V(pi_e):                  {V_dr:+.4f}")
print(f"Ground truth V(pi_e):                    {r_cand.mean():+.4f}")
print(f"DR bias vs ground truth:                 {V_dr - r_cand.mean():+.4f}")

## Part 6 — Side-by-side comparison

In [ ]:
table = pd.DataFrame({
    "estimator":  ["Naive logged mean", "IPS", "SNIPS", "DR", "Ground truth (candidate)"],
    "estimate":   [r_log.mean(), V_ips, V_snips, V_dr, r_cand.mean()],
    "bias":       [r_log.mean() - r_cand.mean(),
                   V_ips - r_cand.mean(),
                   V_snips - r_cand.mean(),
                   V_dr - r_cand.mean(),
                   0.0],
})
print(table.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))

**Read the table.**

- **Naive logged mean** is what an analyst who ignored the policy difference would report — it averages reward under the *mixed* behaviour policy and is the wrong target.
- **IPS** is unbiased in expectation but high-variance when the ESS is small.
- **SNIPS** trades a small bias for sharper variance, useful when IPS has wide confidence intervals.
- **DR** uses the fitted reward model to cut variance further; if the model fits well, DR is the most efficient unbiased estimator.

The bias of each estimator relative to the ground-truth candidate return tells us whether OPE on the logged data can substitute for actually deploying the candidate.

## Part 7 — Diagnostics: when to *not* trust OPE

OPE's reliability depends on three things, each with a simple diagnostic:

1. **Effective sample size.** ESS / N below ~10% means IPS variance is dominating any signal. Report SNIPS or DR instead, but also note that the candidate policy is reaching states the logged policy rarely visited.
2. **Reward-model fit.** Check the cross-validated MSE of $\hat{r}$ on logged data. A bad fit means DR's variance-reduction promise is weak.
3. **Coverage.** If $\pi_e$ chooses actions $\pi_b$ never took in some states, no OPE estimator is valid there. For us this is trivially fine (both actions appear with prob 0.5), but state-conditioned $\pi_e$ on real data often violates coverage.

In [ ]:
from sklearn.model_selection import cross_val_score

mse = -cross_val_score(GradientBoostingRegressor(random_state=0), features, r_log,
                       scoring="neg_mean_squared_error", cv=5).mean()
r_var = r_log.var()
print(f"Reward-model 5-fold MSE:    {mse:.6f}")
print(f"Variance of logged reward:  {r_var:.6f}")
print(f"R^2 of fit:                 {1 - mse/r_var:.3f}")
print()
print(f"ESS fraction (from Part 3): {100*ess_frac:.1f}%")

## Part 8 — Decision

Three bullets, the deployment-readiness report:

1. **Best OPE estimate of the candidate policy's expected per-step reward is the DR number** (read from Part 6's table); the bias against ground truth is the residual gap between OPE and a live pilot.

2. **The OPE estimate is trustworthy / fragile** based on the ESS fraction and reward-model $R^2$ from Part 7. ESS > 25% and $R^2$ > 0.5 is the comfortable regime; below either, treat the OPE number as an order-of-magnitude check rather than a deployment-grade estimate.

3. **If the bias of all three (IPS, SNIPS, DR) against ground truth is small, OPE is doing the job we hoped — substituting for a live pilot.** If the bias is large *and* the diagnostics looked fine, then either the reward function is poorly chosen, or the per-step IPS form is inadequate (the candidate policy's state distribution diverges from the logged distribution after a few steps, breaking the i.i.d. assumption per-step IPS makes).

## Reflection

**The TE simulator is its own oracle.** Most real OPE problems give us only the logged data; we have to take the OPE estimate on faith. Here we can directly compute $V(\pi_e)$ from the candidate trajectory, and the comparison teaches us *which* OPE estimators are reliable in this regime.

**The biggest pitfall in industrial OPE is silent extrapolation.** When $\pi_e$ takes the system into states $\pi_b$ rarely visited, every OPE estimator extrapolates from the model implicit in the data — and extrapolation in chemical process control is where catastrophes live. The ESS diagnostic flags this in numbers; the chapter's Markov-coverage check (Lab 11A) flags it structurally. Run both.

## What's next

Lab 12B uses the same TE simulator to build a *learned digital twin* — fit a structural model of the process from logged data, then plan a policy against the twin and validate against the real simulator. The twin step is what enables genuine causal reinforcement learning rather than purely observational OPE.